# IS 507 原始資料驗證與查詢時前處理

本專案保留 Alibaba 官方 Parquet 原始檔，不另外建立一份完整的 processed 副本。本 notebook 有三個目標：

1. 確認預期的原始資料分區與欄位結構是否完整；
2. 在分析前記錄缺失值與例外值；
3. 使用 DuckDB view 或查詢，示範可還原且不修改原始檔的清理規則。

欄位字典請參考 `DATA_README.md`；組員使用 R 讀取資料的範例請參考 `r_start.R`。

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "MID" / "data").exists():
    ROOT = ROOT / "MID"

DATA_DIR = ROOT / "data"
POD_ROOT = DATA_DIR / "asi_opensource_pod_hourly"
POD_GLOB = (POD_ROOT / "day=*" / "hour=*" / "part-000.parquet").as_posix()
JOB_PARQUET = DATA_DIR / "asi_opensource_job_execution_summary" / "part-000.parquet"
MERGED_PARQUET = DATA_DIR / "asi_opensource_pod_hourly_day0_29.parquet"
if not JOB_PARQUET.exists():  # 相容先前將檔案直接放在 data/ 的目錄配置
    JOB_PARQUET = DATA_DIR / "part-000.parquet"

assert POD_ROOT.exists(), f"找不到 pod 資料夾：{POD_ROOT}"
assert JOB_PARQUET.exists(), f"找不到 execution summary Parquet：{JOB_PARQUET}"
print("Pod 資料位置：", POD_ROOT)
print("Execution summary 位置：", JOB_PARQUET)

Pod 資料位置： c:\Users\ethan\Desktop\UIUC26Fall\IS507\MID\data\asi_opensource_pod_hourly
Execution summary 位置： c:\Users\ethan\Desktop\UIUC26Fall\IS507\MID\data\asi_opensource_job_execution_summary\part-000.parquet


## 1. 下載狀態與分區清單

目前規劃的資料範圍是 `day=0..29`，每天應有 24 個小時分區。下載仍在進行時也可以安全執行此儲存格；它只會列出尚未完整的日期，不會中止整份 notebook。

In [2]:
pod_files = sorted(POD_ROOT.glob("day=*/hour=*/part-000.parquet"))
records = []
for path in pod_files:
    records.append({
        "day": int(path.parents[1].name.split("=")[1]),
        "hour": int(path.parent.name.split("=")[1]),
        "bytes": path.stat().st_size,
    })

inventory_raw = pd.DataFrame(records, columns=["day", "hour", "bytes"])
inventory = pd.DataFrame({"day": range(30)})
if not inventory_raw.empty:
    by_day = inventory_raw.groupby("day").agg(
        parquet_files=("hour", "size"),
        unique_hours=("hour", "nunique"),
        bytes=("bytes", "sum"),
    )
    inventory = inventory.join(by_day, on="day")
inventory[["parquet_files", "unique_hours", "bytes"]] = (
    inventory[["parquet_files", "unique_hours", "bytes"]].fillna(0).astype("int64")
)
inventory["gib"] = inventory["bytes"] / 1024**3
inventory["complete"] = inventory["unique_hours"].eq(24)
display(inventory)
print(f"已完成日期：{inventory['complete'].sum()}/30")

,day,parquet_files,unique_hours,bytes,gib,complete
0,0,24,24,1521336922,1.416855,True
1,1,24,24,1431666015,1.333343,True
2,2,24,24,1457954257,1.357826,True
3,3,24,24,1385278793,1.290141,True
4,4,24,24,1469045000,1.368155,True
5,5,24,24,1469036796,1.368147,True
6,6,24,24,1508969635,1.405337,True
7,7,24,24,1512506528,1.408631,True
8,8,24,24,1539918291,1.434161,True
9,9,24,24,1475310966,1.373990,True


已完成日期：17/30


## 2. 單一 Parquet 的 metadata 完整性檢查

這個區段先讀取 Parquet footer，不會將 8.4 億列全部載入記憶體。它會比較 720 個來源檔與合併檔的 row count，並確認合併檔含有 25 個欄位，其中 `day` 與 `hour` 已成為實體整數欄位。

合併程序只做欄位重組與壓縮，沒有刪除 rows、移除 `workload_id`、補值或清除缺失值。

In [ ]:
import pyarrow.parquet as pq

assert MERGED_PARQUET.exists(), (
    f"找不到合併檔：{MERGED_PARQUET}。請先執行 python merge_pod_hourly.py"
)

# 從每個來源檔的 footer 取得 row count，不需要掃描全部欄位。
source_partition_rows = []
for path in pod_files:
    day = int(path.parents[1].name.split("=")[1])
    hour = int(path.parent.name.split("=")[1])
    rows = pq.ParquetFile(path).metadata.num_rows
    source_partition_rows.append({"day": day, "hour": hour, "source_rows": rows})

source_partition_rows = pd.DataFrame(source_partition_rows).sort_values(["day", "hour"])
source_total_rows = int(source_partition_rows["source_rows"].sum())

merged_file = pq.ParquetFile(MERGED_PARQUET)
merged_total_rows = merged_file.metadata.num_rows
merged_schema = merged_file.schema_arrow

merged_metadata_summary = pd.DataFrame({
    "metric": [
        "file_size_gib", "row_groups", "columns",
        "source_rows", "merged_rows", "row_difference"
    ],
    "value": [
        MERGED_PARQUET.stat().st_size / 1024**3,
        merged_file.metadata.num_row_groups,
        len(merged_schema.names),
        source_total_rows,
        merged_total_rows,
        merged_total_rows - source_total_rows,
    ],
})
display(merged_metadata_summary)

schema_table = pd.DataFrame({
    "column": merged_schema.names,
    "type": [str(field.type) for field in merged_schema],
})
display(schema_table)

assert len(pod_files) == 720
assert merged_total_rows == source_total_rows
assert merged_total_rows == 842_390_418
assert len(merged_schema.names) == 25
assert str(merged_schema.field("day").type) == "int16"
assert str(merged_schema.field("hour").type) == "int8"
print("Metadata 驗證通過：720 個來源檔與合併檔 row count 完全一致。")

## 3. 建立唯讀 DuckDB views

DuckDB 會直接查詢原始 Parquet 檔，不需要先把所有資料載入記憶體。`pod_raw` 與 `job_raw` 保留來源檔的原始欄位；`day` 和 `hour` 則由 Hive 格式的資料夾名稱自動推斷。

In [3]:
con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='8GB'")
con.execute("SET preserve_insertion_order=false")

con.execute(f'''\
CREATE OR REPLACE VIEW pod_raw AS
SELECT *
FROM read_parquet('{POD_GLOB}', hive_partitioning=true, union_by_name=true)
''')
con.execute(f'''\
CREATE OR REPLACE VIEW job_raw AS
SELECT *
FROM read_parquet('{JOB_PARQUET.as_posix()}')
''')

con.execute(f'''\
CREATE OR REPLACE VIEW pod_merged AS
SELECT *
FROM read_parquet('{MERGED_PARQUET.as_posix()}')
''')

print("pod_raw 欄位結構")
display(con.execute("DESCRIBE pod_raw").df())
print("job_raw 欄位結構")
display(con.execute("DESCRIBE job_raw").df())

pod_raw 欄位結構


,column_name,column_type,null,key,default,extra
0,pod_id,VARCHAR,YES,None,None,None
1,workload_id,VARCHAR,YES,None,None,None
2,server_id,VARCHAR,YES,None,None,None
3,cluster_id,VARCHAR,YES,None,None,None
4,state_public,VARCHAR,YES,None,None,None
5,gpu_spec_public,VARCHAR,YES,None,None,None
6,server_gpu_count,BIGINT,YES,None,None,None
7,server_cpu_capacity_cores,BIGINT,YES,None,None,None
8,priority_class,VARCHAR,YES,None,None,None
9,job_type_public,VARCHAR,YES,None,None,None


job_raw 欄位結構


,column_name,column_type,null,key,default,extra
0,pod_id,VARCHAR,YES,None,None,None
1,workload_id,VARCHAR,YES,None,None,None
2,server_id,VARCHAR,YES,None,None,None
3,gpu_spec_public,VARCHAR,YES,None,None,None
4,priority_class,VARCHAR,YES,None,None,None
5,job_type_public,VARCHAR,YES,None,None,None
6,model_type_public,VARCHAR,YES,None,None,None
7,is_genai_request,BOOLEAN,YES,None,None,None
8,gpu_request,DOUBLE,YES,None,None,None
9,duration_hours,DOUBLE,YES,None,None,None


## 3.1 合併內容與分區 row count 驗證

這個區段確認 `day` 與 `hour` coverage，並逐一比較 720 個 day/hour partitions 的來源 row count 與合併後 row count。最後再比較 day 0 的代表性缺失值與例外值計數，確認合併過程沒有執行清值。

查詢只回傳小型彙總表；不會把完整資料轉成 pandas DataFrame。

In [ ]:
merged_coverage = con.execute('''
SELECT
    count(*) AS rows,
    min(day) AS min_day,
    max(day) AS max_day,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    count(DISTINCT day) AS distinct_days,
    count(DISTINCT hour) AS distinct_hours
FROM pod_merged
''').df()
display(merged_coverage)

coverage = merged_coverage.iloc[0]
assert int(coverage["rows"]) == source_total_rows
assert (int(coverage["min_day"]), int(coverage["max_day"])) == (0, 29)
assert (int(coverage["min_hour"]), int(coverage["max_hour"])) == (0, 23)
assert int(coverage["distinct_days"]) == 30
assert int(coverage["distinct_hours"]) == 24

merged_partition_rows = con.execute('''
SELECT day, hour, count(*) AS merged_rows
FROM pod_merged
GROUP BY day, hour
ORDER BY day, hour
''').df()

partition_comparison = source_partition_rows.merge(
    merged_partition_rows,
    on=["day", "hour"],
    how="outer",
    validate="one_to_one",
)
partition_comparison["row_difference"] = (
    partition_comparison["merged_rows"] - partition_comparison["source_rows"]
)
display(partition_comparison)

assert len(partition_comparison) == 720
assert partition_comparison["row_difference"].eq(0).all()
print("Partition 驗證通過：720 個 day/hour row counts 全部一致。")

# 比較代表性統計，確認 merge 沒有清除 NULL、零值或負值。
day0_preservation = con.execute('''
SELECT 'source_day0' AS table_name,
       count(*) AS rows,
       count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
       count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_sm_util,
       count(*) FILTER (WHERE gpu_request = 0) AS zero_gpu_request,
       count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
       count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM pod_raw WHERE day = 0
UNION ALL
SELECT 'merged_day0', count(*),
       count(*) FILTER (WHERE workload_id IS NULL),
       count(*) FILTER (WHERE avg_gpu_sm_util IS NULL),
       count(*) FILTER (WHERE gpu_request = 0),
       count(*) FILTER (WHERE schedule_delay_sec < 0),
       count(*) FILTER (WHERE ready_delay_sec < 0)
FROM pod_merged WHERE day = 0
''').df()
display(day0_preservation)

assert day0_preservation.iloc[0, 1:].equals(day0_preservation.iloc[1, 1:])
print("內容抽查通過：day 0 的 NULL、零值與負值計數在合併前後一致。")

## 4. 可還原的查詢時資料清理

這裡不會覆寫原始值，也不會輸出新的大型資料檔。

- 保留官方欄位名稱。
- 只有研究問題針對 GPU workload 時，才篩選 `gpu_request > 0`。
- 保留缺失值；不全面刪除含缺失值的資料列，也不直接以平均數補值。
- 字面值 `Unknown`／`unknown` 是有效類別，不是 null。
- 負的 delay 可在名稱明確的 `_clean` 衍生欄位中改為缺失值。
- 保留原始 `avg_gpu_sm_util`，不將數值強制截斷在 100。
- 官方 schema 沒有說明 `gpu_mem_request` 的單位，因此保留來源值，不自行轉換。

In [4]:
con.execute('''
CREATE OR REPLACE VIEW pod_analysis AS
SELECT
    *,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request
FROM pod_raw
''')

con.execute('''
CREATE OR REPLACE VIEW job_analysis AS
SELECT
    *,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean
FROM job_raw
''')

print("已建立記憶體內 views：pod_analysis、job_analysis（未輸出新檔案）")

已建立記憶體內 views：pod_analysis、job_analysis（未輸出新檔案）


## 5. 資料品質檢查

預設先檢查一個已下載完成的日期，以縮短執行時間。需要更完整的稽核時，可以修改 `CHECK_DAYS`。以下統計只檢查缺失值與數值範圍，不會建立新的資料檔。

In [5]:
available_days = inventory.loc[inventory["complete"], "day"].tolist()
if not available_days:
    raise RuntimeError("目前還沒有下載完整的 pod 日期。")

# 預設只檢查第一個完整日期；需要時可改成例如 [0, 1, 2]。
CHECK_DAYS = [available_days[0]]
days_sql = ", ".join(map(str, CHECK_DAYS))

pod_quality = con.execute(f'''\
SELECT
    count(*) AS rows,
    count(DISTINCT pod_id) AS unique_pods,
    count(DISTINCT workload_id) AS unique_nonnull_workloads,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_avg_gpu_sm_util,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay,
    min(gpu_request) AS min_gpu_request,
    max(gpu_request) AS max_gpu_request,
    min(used_gpu_hours) AS min_used_gpu_hours,
    max(used_gpu_hours) AS max_used_gpu_hours,
    min(avg_gpu_sm_util) AS min_avg_gpu_sm_util,
    max(avg_gpu_sm_util) AS max_avg_gpu_sm_util
FROM pod_raw
WHERE day IN ({days_sql})
''').df()
print(f"Pod 資料品質摘要（日期：{CHECK_DAYS}）")
display(pod_quality)

job_quality = con.execute('''
SELECT
    count(*) AS rows,
    count(DISTINCT pod_id) AS unique_pods,
    count(DISTINCT workload_id) AS unique_nonnull_workloads,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE duration_hours IS NULL) AS missing_duration,
    count(*) FILTER (WHERE duration_hours < 0) AS negative_duration,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay,
    min(duration_hours) AS min_duration_hours,
    max(duration_hours) AS max_duration_hours
FROM job_raw
''').df()
print("Execution summary 資料品質摘要")
display(job_quality)

Pod 資料品質摘要（日期：[0]）


,rows,unique_pods,unique_nonnull_workloads,missing_workload_id,missing_avg_gpu_sm_util,missing_schedule_delay,negative_schedule_delay,negative_ready_delay,min_gpu_request,max_gpu_request,min_used_gpu_hours,max_used_gpu_hours,min_avg_gpu_sm_util,max_avg_gpu_sm_util
0,28063089,8261309,117751,6857916,26310926,849102,4848,7400127,0.0,16.0,0.0,16.0,0.0,1187.06519


Execution summary 資料品質摘要


,rows,unique_pods,unique_nonnull_workloads,missing_workload_id,missing_duration,negative_duration,negative_schedule_delay,negative_ready_delay,min_duration_hours,max_duration_hours
0,40522321,36685018,501183,35729919,0,0,232953,22502335,0.005556,1824.0


## 6. 不產生 processed 檔案的分析範例

將查詢結果轉成 pandas DataFrame 前，應先完成篩選或彙總。第一個查詢只讀取一個小時並限制為 1,000 列；第二個查詢則將目前已下載的分區彙總成較小的每日結果。

In [6]:
example_hour = con.execute('''
SELECT pod_id, workload_id, gpu_spec_public, priority_class,
       job_type_public, gpu_request, used_gpu_hours, avg_gpu_sm_util,
       ready_status, day, hour
FROM pod_analysis
WHERE day = 0 AND hour = 0 AND is_gpu_request
LIMIT 1000
''').df()
print("Day 0、hour 0 的 GPU-requesting pod 範例")
display(example_hour.head())

daily_gpu = con.execute('''
SELECT day,
       count(*) AS rows,
       count(DISTINCT pod_id) AS unique_pods,
       sum(gpu_request) AS requested_gpu_hours,
       sum(used_gpu_hours) AS used_gpu_hours
FROM pod_analysis
WHERE is_gpu_request
GROUP BY day
ORDER BY day
''').df()
print("各日期 GPU 使用摘要")
display(daily_gpu)

Day 0、hour 0 的 GPU-requesting pod 範例


,pod_id,workload_id,gpu_spec_public,priority_class,job_type_public,gpu_request,used_gpu_hours,avg_gpu_sm_util,ready_status,day,hour
0,7d89e44ff3e215de1d540cefbf8407f2,NaN,H800,LP,offline_inference,2.00,0.533333,85.914383,False,0,00
1,cd20dfab59fc896c29f2953d0953b511,NaN,A10,HP,online_inference,0.11,0.110000,NaN,False,0,00
2,3cb33a471ebeec90ae4807866550b0b9,33d2c5baf5bb0690947fa2577bf9561e,XPU-E,HP,online_inference,1.00,1.000000,2.127781,True,0,00
3,4ac0b404b5c01c1e51e117cfe17343c7,NaN,XPU-E,LP,offline_inference,1.00,0.038889,62.612783,True,0,00
4,374a9570adf62971137478d8ab8d49c5,19345577928eb58c1d4e6a8a93b99475,XPU-E,HP,training,0.50,0.177778,NaN,True,0,00


各日期 GPU 使用摘要


,day,rows,unique_pods,requested_gpu_hours,used_gpu_hours
0,0,2259971,400380,2.810003e+06,2.410951e+06
1,1,2188309,355902,2.771447e+06,2.421660e+06
2,2,2128549,286470,2.712207e+06,2.441932e+06
3,3,2105772,264887,2.666473e+06,2.430058e+06
4,4,2122461,295934,2.740040e+06,2.439287e+06
5,5,1949862,241028,2.534536e+06,2.296246e+06
6,6,1992364,231146,2.575105e+06,2.345863e+06
7,7,2137965,319393,2.737258e+06,2.427072e+06
8,8,2328273,398965,2.951060e+06,2.559651e+06
9,9,2337063,352491,2.928128e+06,2.615980e+06


## 未來合併兩份資料時的原則

Pod-hourly 與 execution-summary 的資料粒度不同，而且兩者的 identifier 直接重疊比例有限，因此不要直接將所有 raw rows 做 many-to-many join。若研究需要合併兩份資料：

1. 先明確定義分析單位，通常可使用 `workload_id`；
2. 在查詢中分別將兩份資料彙總成每個非缺失 `workload_id` 一列；
3. 再合併這兩個較小的彙總結果；
4. 報告成功配對與未配對的 workload 數量及比例。

如果研究主題是 execution readiness 或 duration，也可以只使用 summary table。建立模型時，建議依 `workload_id` 分割 training/test data，以降低相關 executions 同時出現在訓練集與測試集所造成的 data leakage。